In [ ]:
# C3. Data Merging

## Objective

The objective of this notebook is to merge the twenty operational PEM fuel cell datasets into a single master dataset while preserving the identity of each original experiment using an `operating_hour` column.

The merged dataset will be saved as `operational_merged_raw.csv` in the `data/processed/` folder.

In [ ]:
## Workflow

This notebook covers the following tasks:

1. Import required libraries
2. Define project paths
3. Load operational datasets
4. Sort files by operating hour
5. Extract operating-hour values from filenames
6. Add an `operating_hour` column to each dataset
7. Merge all operational datasets
8. Verify merged dataset dimensions
9. Validate that operating-hour labels were assigned correctly
10. Save the merged dataset
11. Record summary and observations

In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

from pathlib import Path
import re

import pandas as pd
import numpy as np

In [2]:
# ============================================================
# Configure Display Settings
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", None)

In [3]:
# ============================================================
# Define Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

print("Project Root:", PROJECT_ROOT)
print("Raw Data Folder:", RAW_DATA_DIR)
print("Processed Data Folder:", PROCESSED_DATA_DIR)

Project Root: C:\Users\usman\Desktop\PEMFC_Dissertation
Raw Data Folder: C:\Users\usman\Desktop\PEMFC_Dissertation\data\raw
Processed Data Folder: C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed


In [ ]:
## C3.1 Locate Operational Dataset Files

Only the operational datasets ending with `_h.csv` are selected for merging. Polarization curve datasets are excluded because they have a different structure and will be analysed separately.

In [4]:
# ============================================================
# Locate Operational Dataset Files
# ============================================================

operational_files = list(RAW_DATA_DIR.glob("*_h.csv"))

print(f"Number of operational files found: {len(operational_files)}")

for file in operational_files:
    print(file.name)

Number of operational files found: 20
1000_h.csv
100_h.csv
150_h.csv
200_h.csv
250_h.csv
300_h.csv
350_h.csv
400_h.csv
450_h.csv
500_h.csv
50_h.csv
550_h.csv
600_h.csv
650_h.csv
700_h.csv
750_h.csv
800_h.csv
850_h.csv
900_h.csv
950_h.csv


In [ ]:
## C3.2 Sort Operational Datasets

The files are currently arranged alphabetically rather than chronologically.

To accurately represent the degradation process of the PEM fuel cell, the datasets must be sorted according to their operating-hour values before merging.

This ensures that the merged dataset follows the correct experimental sequence from 50 hours to 1000 hours.

In [5]:
# ============================================================
# Sort Operational Files by Operating Hour
# ============================================================

# Extract the operating hour from each filename
operational_files = sorted(
    operational_files,
    key=lambda file: int(re.search(r"(\d+)_h", file.stem).group(1))
)

print("Operational datasets in chronological order:\n")

for file in operational_files:
    print(file.name)

Operational datasets in chronological order:

50_h.csv
100_h.csv
150_h.csv
200_h.csv
250_h.csv
300_h.csv
350_h.csv
400_h.csv
450_h.csv
500_h.csv
550_h.csv
600_h.csv
650_h.csv
700_h.csv
750_h.csv
800_h.csv
850_h.csv
900_h.csv
950_h.csv
1000_h.csv


In [ ]:
## C3.3 Extract Operating Hour and Merge Operational Datasets

Each operational dataset represents measurements collected after a specific operating period (e.g., 50 hours, 100 hours, or 1000 hours).

Before merging the datasets, a new variable named **`operating_hour`** is added to every dataset. This variable preserves the origin of each observation after all datasets are combined into a single master dataset.

The datasets are then merged in chronological order to create one unified dataset for subsequent analysis.

In [6]:
# ============================================================
# Extract Operating Hour and Merge Operational Datasets
# ============================================================

# List to store all operational DataFrames
merged_datasets = []

# Process each operational dataset
for file in operational_files:

    # Read dataset
    df = pd.read_csv(file, low_memory=False)

    # Extract operating hour from filename
    operating_hour = int(re.search(r"(\d+)_h", file.stem).group(1))

    # Add new column
    df["operating_hour"] = operating_hour

    # Store dataset
    merged_datasets.append(df)

# Merge all datasets
merged_dataset = pd.concat(
    merged_datasets,
    ignore_index=True
)

print("Operational datasets merged successfully.")

Operational datasets merged successfully.


In [12]:
# ============================================================
# Move operating_hour to the First Column
# ============================================================

columns = ["operating_hour"] + [
    col for col in merged_dataset.columns
    if col != "operating_hour"
]

merged_dataset = merged_dataset[columns]

In [ ]:
## C3.4 Merge Validation

This section verifies that the merge was performed correctly. The validation checks confirm that all rows from the original operational datasets are present in the merged dataset and that the `operating_hour` column correctly preserves the source experiment for each observation.

In [7]:
# ============================================================
# C3.4 Merge Validation: Row Count Verification
# ============================================================

# Expected total rows from original files
expected_total_rows = 0

for file in operational_files:
    df_temp = pd.read_csv(file, low_memory=False)
    expected_total_rows += df_temp.shape[0]

# Actual total rows after merging
actual_total_rows = merged_dataset.shape[0]

print("Expected total rows:", expected_total_rows)
print("Actual total rows:", actual_total_rows)
print("Row count match:", expected_total_rows == actual_total_rows)

Expected total rows: 3629720
Actual total rows: 3629720
Row count match: True


In [8]:
# ============================================================
# C3.4 Merge Validation: Operating Hour Counts
# ============================================================

operating_hour_counts = (
    merged_dataset["operating_hour"]
    .value_counts()
    .sort_index()
    .reset_index()
)

operating_hour_counts.columns = ["operating_hour", "row_count"]

operating_hour_counts

,operating_hour,row_count
0,50,179362
1,100,179362
2,150,179362
3,200,179362
4,250,179362
5,300,179362
6,350,179362
7,400,179362
8,450,179362
9,500,179362


In [ ]:
## C3.4 Merge Validation – Data Integrity Verification

The merged dataset should preserve every observation exactly as it appears in the original operational datasets.

To verify this, we compare one original dataset with the corresponding subset extracted from the merged dataset.

If both datasets are identical (apart from the additional `operating_hour` column), we can conclude that the merge preserved the original data without modification.

In [9]:
# ============================================================
# Verify Data Integrity Using the 50_h Dataset
# ============================================================

# Original dataset
original_50 = pd.read_csv(
    RAW_DATA_DIR / "50_h.csv",
    low_memory=False
)

# Extract 50-hour observations from merged dataset
merged_50 = (
    merged_dataset[merged_dataset["operating_hour"] == 50]
    .drop(columns="operating_hour")
    .reset_index(drop=True)
)

# Compare
datasets_match = original_50.equals(merged_50)

print("Original and merged datasets are identical:", datasets_match)

Original and merged datasets are identical: True


In [ ]:
## C3.4 Merge Validation – Boundary Verification

The final verification checks the transition between consecutive operating-hour datasets.

This confirms that the observations from one experiment end correctly before the observations from the next experiment begin.

In [13]:
# ============================================================
# Boundary Verification
# ============================================================

print("Last 3 rows of the 50-hour dataset")
display(
    merged_dataset[
        merged_dataset["operating_hour"] == 50
    ].tail(3)
)

print("\nFirst 3 rows of the 100-hour dataset")
display(
    merged_dataset[
        merged_dataset["operating_hour"] == 100
    ].head(3)
)

Last 3 rows of the 50-hour dataset


,operating_hour,Unnamed: 0,time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow
179359,50,179359,179358.761,0,0.9352,0,109.496603,109.72064,103.628455,102.80437,83.18354,55.065323,70.514626,42.237297,64.997787,70.139465,59.94846,0.07,0.291
179360,50,179360,179359.761,0,0.9356,0,109.395428,109.619538,103.628455,103.108188,83.269958,55.013645,70.565132,42.210842,64.972542,70.063713,59.987217,0.07,0.291
179361,50,179361,179360.761,0,0.9349,0,109.496603,109.72064,103.628455,103.006915,83.247559,54.948143,70.509048,42.238628,64.93663,69.951378,59.866337,0.07,0.291



First 3 rows of the 100-hour dataset


,operating_hour,Unnamed: 0,time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow
179362,100,0,s,A,V,W,kPag,kPag,kPag,kPag,°C,°C,°C,°C,°C,°C,°C,NLPM,NLPM
179363,100,1,Elapsed time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow
179364,100,2,1.226,0,0.9433,0,109.901303,110.529453,109.395428,108.792114,83.004883,55.516544,69.586266,37.81496,64.555069,70.225243,56.26281,0.07,0.291


In [ ]:
## C3.5 Save the Merged Dataset

After validating the merged dataset, it is saved as a new file in the `data/processed/` directory.

Saving the merged dataset ensures that subsequent stages of the project can use a single unified dataset without repeatedly merging the original operational datasets.

The original datasets stored in the `data/raw/` directory remain unchanged throughout the project.

In [14]:
# ============================================================
# Save Merged Dataset
# ============================================================

output_file = PROCESSED_DATA_DIR / "operational_merged_raw.csv"

merged_dataset.to_csv(output_file, index=False)

print("Merged dataset saved successfully.")
print(f"Location: {output_file}")

Merged dataset saved successfully.
Location: C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed\operational_merged_raw.csv


In [ ]:
## C3.6 Summary and Initial Observations

In [ ]:
### Summary

The twenty operational PEM fuel cell datasets were successfully merged into a single master dataset while preserving the identity of each experiment through the addition of an `operating_hour` variable.

Several validation checks confirmed that the merge was performed correctly. The total number of observations in the merged dataset matched the combined number of observations in the original datasets, and each operating hour contained the expected number of records. Data integrity verification further demonstrated that the merged dataset preserved the original experimental observations without modification.

During validation, it was also observed that each operational dataset contains two metadata rows immediately after the header row. These rows provide measurement units and variable descriptions rather than experimental observations. As a result, many operational variables are currently stored as `object` data types instead of numerical types.

No cleaning or preprocessing has been performed at this stage. The merged dataset therefore remains an exact representation of the original experimental data and has been saved for use in the subsequent stages of the project.

### Initial Observations

- Twenty operational datasets were successfully merged.
- Experimental identities were preserved using the `operating_hour` variable.
- The chronological order of the experiments was maintained from 50 hours to 1000 hours.
- The merged dataset passed all merge validation checks.
- Two metadata rows were identified within each operational dataset.
- Operational variables currently have `object` data types due to the presence of metadata rows.
- No modifications have been made to the original raw datasets.
- The merged dataset is ready for initial data understanding and subsequent data cleaning.